In [ ]:
# after lightglue, before ransac

keypoints = []

for (a, b), data in matches.items():
    idx = data["idx"]
    kp0, kp1 = feats[a]["keypoints"], feats[b]["keypoints"]

    matches0 = np.full(len(kp0), -1, dtype=np.int32)
    matches0[idx[:, 0]] = idx[:, 1]

    scores0 = np.zeros(len(kp0), dtype=np.float32)
    scores0[idx[:, 0]] = data["scores"]

    keypoints.append({
        "dataset": "provided_features",
        "scene": "provided_scene",
        "pair": (a, b),
        "keypoints0": kp0,
        "keypoints1": kp1,
        "matches0": matches0,
        "scores0": scores0
    })

In [ ]:
# Filter image pairs using RANSAC statistics

# Hyperparameters
min_matches = 15
min_inliers = 12
min_inlier_ratio = 0.20

# Keep pairs satisfying all three conditions
filtered_pairs = [
    result for result in ransac_results
    if result["num_matches"] >= min_matches
    and result["num_inliers"] >= min_inliers
    and result["inlier_ratio"] >= min_inlier_ratio
]

# Summary
print(f"Total pairs before filtering: {len(ransac_results)}")
print(f"Pairs retained: {len(filtered_pairs)}")
print(f"Pairs rejected: {len(ransac_results) - len(filtered_pairs)}")

if ransac_results:
    print(
        f"Retention rate: "
        f"{len(filtered_pairs) / len(ransac_results):.1%}"
    )

# Display the strongest retained pairs
filtered_pairs_sorted = sorted(
    filtered_pairs,
    key=lambda r: (r["num_inliers"], r["inlier_ratio"]),
    reverse=True
)

print("\nTop retained pairs:")

for result in filtered_pairs_sorted[:15]:
    print(
        f'{result["dataset"]} | '
        f'{result["pair"][0]} <-> {result["pair"][1]} | '
        f'Matches: {result["num_matches"]} | '
        f'Inliers: {result["num_inliers"]} | '
        f'Ratio: {result["inlier_ratio"]:.1%}'
    )


In [ ]:
%pip install networkx
import networkx as nx
import numpy as np

image_graph = nx.Graph()

for result in filtered_pairs:
    dataset = result["dataset"]
    scene = result["scene"]
    image_a, image_b = result["pair"]

    node_a = (dataset, image_a)
    node_b = (dataset, image_b)

    kp0 = np.asarray(result["keypoints0"], dtype=np.float32)
    kp1 = np.asarray(result["keypoints1"], dtype=np.float32)
    matches0 = np.asarray(result["matches0"], dtype=np.int32)
    inlier_mask = np.asarray(result["inlier_mask"], dtype=bool)

    # Recover the original SuperPoint indices for valid matches.
    matched_indices0 = np.flatnonzero(matches0 >= 0)
    matched_indices1 = matches0[matched_indices0]

    if len(inlier_mask) != len(matched_indices0):
        raise ValueError(
            f"Inlier mask mismatch for pair {(image_a, image_b)}"
        )

    # Keep only RANSAC inliers, expressed as original keypoint indices.
    inlier_matches = np.column_stack((
        matched_indices0[inlier_mask],
        matched_indices1[inlier_mask]
    )).astype(np.uint32)

    # Store each image's original keypoints once.
    for node, keypoints_array in (
        (node_a, kp0),
        (node_b, kp1),
    ):
        if node not in image_graph:
            image_graph.add_node(
                node,
                dataset=dataset,
                scene=scene,
                image_name=node[1],
                keypoints=keypoints_array,
            )
        else:
            existing = image_graph.nodes[node]["keypoints"]

            # The same image must always use the same keypoint array
            # and therefore the same feature indices.
            if (
                existing.shape != keypoints_array.shape
                or not np.allclose(existing, keypoints_array, atol=1e-4)
            ):
                raise ValueError(
                    f"Inconsistent keypoints for image {node}. "
                    "Check feature caching and image ordering."
                )

    # Preserve the original pair order explicitly.
    # inlier_matches[:, 0] indexes keypoints0 for image_a;
    # inlier_matches[:, 1] indexes keypoints1 for image_b.
    image_graph.add_edge(
        node_a,
        node_b,
        image0=node_a,
        image1=node_b,
        num_matches=int(result["num_matches"]),
        num_inliers=int(result["num_inliers"]),
        inlier_ratio=float(result["inlier_ratio"]),
        fundamental_matrix=result["fundamental_matrix"],
        inlier_matches=inlier_matches,
    )

print("COLMAP-ready image graph built.")
print(f"Images (nodes): {image_graph.number_of_nodes()}")
print(f"Verified pairs (edges): {image_graph.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(image_graph)}")

components = sorted(
    nx.connected_components(image_graph),
    key=len,
    reverse=True
)

print(f"\nConnected components: {len(components)}")
for i, component in enumerate(components[:10], start=1):
    print(f"\nComponent {i}: {len(component)} images")
    for node in sorted(component):
        print(" ", node)


In [ ]:
# sort out the connected components

components = sorted(
    nx.connected_components(image_graph),
    key=len,
    reverse=True
)

print("Connected components:", len(components))

for i, component in enumerate(components[:10], start=1):
    print(f"\nComponent {i}: {len(component)} images")
    for node in sorted(component):
        print(" ", node)